In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
filePath="../Data/customer_churn_dataset-testing-master.csv"
df = pd.read_csv(filePath)
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [ ]:
df.shape

(64374, 12)

In [ ]:
df = df.drop("CustomerID", axis=1)

In [ ]:
df.isnull().sum()

,0
Age,0
Gender,0
Tenure,0
Usage Frequency,0
Support Calls,0
Payment Delay,0
Subscription Type,0
Contract Length,0
Total Spend,0
Last Interaction,0


In [ ]:
df = df.dropna()

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: Index(['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay',
       'Total Spend', 'Last Interaction'],
      dtype='object')
Categorical: Index(['Gender', 'Subscription Type', 'Contract Length'], dtype='object')


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [ ]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

In [ ]:
logistic_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay',
       'Total Spend', 'Last Interaction'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Gender', 'Subscription Type', 'Contract Length'], dtype='object'))])),
                ('classifier', LogisticRegression())])

In [ ]:
y_pred = logistic_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8316893203883495
Precision: 0.8162869607367911
Recall: 0.8306478132193358
F1 Score: 0.8234047754869204
Confusion Matrix:
 [[5656 1137]
 [1030 5052]]


In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, random_state=42))
])

tree_pipeline.fit(X_train, y_train)

y_pred_tree = tree_pipeline.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))
print("Decision Tree Recall:", recall_score(y_test, y_pred_tree))

Decision Tree Accuracy: 0.9596893203883495
Decision Tree Recall: 0.9824071029266689


In [ ]:
print("Training Accuracy (Tree):", tree_pipeline.score(X_train, y_train))
print("Testing Accuracy (Tree):", tree_pipeline.score(X_test, y_test))

Training Accuracy (Tree): 0.9563292491116332
Testing Accuracy (Tree): 0.9596893203883495


In [ ]:


# Get feature names after preprocessing
feature_names = tree_pipeline.named_steps["preprocessor"].get_feature_names_out()

# Get importance values
importances = tree_pipeline.named_steps["classifier"].feature_importances_

# Create dataframe
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# Sort by importance
feature_importance_df = feature_importance_df.sort_values(by="Importance", ascending=False)

feature_importance_df.head(10)

,Feature,Importance
4,num__Payment Delay,0.478695
3,num__Support Calls,0.143954
1,num__Tenure,0.099078
2,num__Usage Frequency,0.091031
7,cat__Gender_Female,0.082769
0,num__Age,0.043059
8,cat__Gender_Male,0.032734
5,num__Total Spend,0.021233
12,cat__Contract Length_Annual,0.007446
6,num__Last Interaction,0.000000


In [ ]:
import joblib

joblib.dump(tree_pipeline, "churn_model.pkl", compress=3)

['churn_model.pkl']

In [ ]:
from google.colab import files
files.download("churn_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1.6.1
